In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import numpy as np
import requests
import zipfile
import io


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


url = "http://mattmahoney.net/dc/text8.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
text = z.read('text8').decode('utf-8').split()


VOCAB_SIZE = 20000
word_counts = Counter(text)
most_common = word_counts.most_common(VOCAB_SIZE - 1)
vocab = {word: i+1 for i, (word, _) in enumerate(most_common)}
vocab["<UNK>"] = 0
idx_to_word = {i: word for word, i in vocab.items()}


text_indices = [vocab.get(word, 0) for word in text]
print(f"Total words: {len(text_indices)}")

Using device: cpu
Total words: 17005207


In [2]:
class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super(SkipGramModel, self).__init__()

        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        self.out_embed = nn.Embedding(vocab_size, embed_dim)

    def forward(self, target, context):


        in_vectors = self.in_embed(target)
        out_vectors = self.out_embed(context)


        return torch.sum(in_vectors * out_vectors, dim=1)

In [8]:
from torch.utils.data import DataLoader, Dataset
import random

class Word2VecDataset(Dataset):
    def __init__(self, text_indices, window_size=4):
        self.text_indices = text_indices
        self.window_size = window_size

    def __len__(self):
        return len(self.text_indices)

    def __getitem__(self, idx):

        target = self.text_indices[idx]


        start = max(0, idx - self.window_size)
        end = min(len(self.text_indices), idx + self.window_size + 1)
        context_candidates = self.text_indices[start:idx] + self.text_indices[idx+1:end]

        if not context_candidates:
            return torch.tensor(target), torch.tensor(target), torch.tensor(target) # Fallback

        context = random.choice(context_candidates)


        negative = random.randint(0, VOCAB_SIZE - 1)

        return torch.tensor(target), torch.tensor(context), torch.tensor(negative)


EMBED_DIM = 100
BATCH_SIZE = 4096
LR = 0.005
EPOCHS = 5

dataset = Word2VecDataset(text_indices)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

model = SkipGramModel(VOCAB_SIZE, EMBED_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()


print("Starting training...")
for epoch in range(EPOCHS):
    total_loss = 0
    for i, (target, context, negative) in enumerate(dataloader):
        target, context, negative = target.to(device), context.to(device), negative.to(device)

        optimizer.zero_grad()


        pos_score = model(target, context)
        pos_loss = criterion(pos_score, torch.ones_like(pos_score))


        neg_score = model(target, negative)
        neg_loss = criterion(neg_score, torch.zeros_like(neg_score))

        loss = pos_loss + neg_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 1000 == 0:
            print(f"Batch {i}, Loss: {loss.item()}")

print("Training Complete!")

Starting training...
Batch 0, Loss: 7.9816694259643555
Batch 1000, Loss: 2.0886027812957764
Batch 2000, Loss: 1.4428131580352783
Batch 3000, Loss: 1.1881654262542725
Batch 4000, Loss: 1.049717664718628
Batch 0, Loss: 1.0097875595092773
Batch 1000, Loss: 0.889650821685791
Batch 2000, Loss: 0.8880887031555176
Batch 3000, Loss: 0.8012605905532837
Batch 4000, Loss: 0.789722204208374
Batch 0, Loss: 0.7636890411376953
Batch 1000, Loss: 0.7325397729873657
Batch 2000, Loss: 0.7285490036010742
Batch 3000, Loss: 0.7016284465789795
Batch 4000, Loss: 0.7034590840339661
Batch 0, Loss: 0.7051926851272583
Batch 1000, Loss: 0.6675387024879456
Batch 2000, Loss: 0.6716835498809814
Batch 3000, Loss: 0.649471640586853
Batch 4000, Loss: 0.6633986830711365
Batch 0, Loss: 0.6749305725097656
Batch 1000, Loss: 0.6758661866188049
Batch 2000, Loss: 0.6624677181243896
Batch 3000, Loss: 0.6724365949630737
Batch 4000, Loss: 0.6644233465194702
Training Complete!


In [5]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 64.9 MB/s eta 0:00:00


In [9]:
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity


gensim_model = api.load("glove-wiki-gigaword-100")


my_embeddings = model.in_embed.weight.data.cpu().numpy()

def get_my_vector(word):
    if word in vocab:
        return my_embeddings[vocab[word]]
    return None


test_word = "queen"
if test_word in vocab:
    my_vec = get_my_vector(test_word).reshape(1, -1)
    gen_vec = gensim_model[test_word].reshape(1, -1)


    my_sim = cosine_similarity(get_my_vector("queen").reshape(1, -1), get_my_vector("king").reshape(1, -1))
    gen_sim = gensim_model.similarity("queen", "king")

    print(f"My Similarity (Queen-King): {my_sim[0][0]}")
    print(f"Gensim Similarity (Queen-King): {gen_sim}")

My Similarity (Queen-King): 0.5969207882881165
Gensim Similarity (Queen-King): 0.7507690787315369


In [18]:
def solve_analogy(a, b, c):

    va, vb, vc = get_my_vector(a), get_my_vector(b), get_my_vector(c)
    result_vec = vb - va + vc


    scores = np.dot(my_embeddings, result_vec)
    best_idx = np.argmax(scores)
    return idx_to_word[best_idx]

print(f"King - Man + Woman = {solve_analogy('man', 'king', 'woman')}")
print(f"France - Paris + Berlin = {solve_analogy('paris', 'france', 'berlin')}")
print(f"Apples - Apple + Car = {solve_analogy('apple', 'apples', 'car')}")
print(f"Small - Big + Hot = {solve_analogy('big', 'small', 'hot')}")
print(f"Sister - Brother + Uncle = {solve_analogy('brother', 'sister', 'uncle')}")
print(f"Walked - Walk + Play = {solve_analogy('walk', 'walked', 'play')}")

King - Man + Woman = meredith
France - Paris + Berlin = silesia
Apples - Apple + Car = apples
Small - Big + Hot = gravel
Sister - Brother + Uncle = cor
Walked - Walk + Play = walked


In [13]:

torch.save(model.state_dict(), "skipgram_model.pth")
print("Model saved successfully!")

from google.colab import files
files.download('skipgram_model.pth')

Model saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_projection_bias(word, gender_direction):

    if word not in vocab:
        return 0.0

    vec = get_my_vector(word).reshape(1, -1)


    score = cosine_similarity(vec, gender_direction.reshape(1, -1))[0][0]
    return score


if 'he' in vocab and 'she' in vocab:
    gender_direction = get_my_vector('he') - get_my_vector('she')


    test_words = ["doctor", "nurse", "mechanic", "teacher", "engineer", "homemaker"]

    print(f"{'WORD':<15} {'BIAS SCORE':<15}")
    print("-" * 30)

    for w in test_words:
        bias = get_projection_bias(w, gender_direction)

        print(f"{w:<15} {bias:.4f}")

else:
    print("Error: 'he' or 'she' not in vocabulary.")

WORD            BIAS SCORE     
------------------------------
doctor          -0.2815
nurse           -0.1419
mechanic        0.1451
teacher         -0.1354
engineer        0.1512
homemaker       0.0000
